# Amazn AI - Day 3
## Retrieval Pipeline & Advanced Search Engine

## Step 1: Import Libraries

In [ ]:
import pandas as pd
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

## Step 2: Load Dataset

In [ ]:
df=pd.read_csv('../data/processed/amazon_cleaned.csv')
df.head()

## Step 3: Chunking & Metadata

In [ ]:
def sentence_splitting(text,max_sentence=2):
    s=sent_tokenize(text)
    return [' '.join(s[i:i+max_sentence]) for i in range(0,len(s),max_sentence)]
all_chunks=[]; metadata_chunks=[]
for _,row in df.iterrows():
    txt=f"{row['product_name']} {row['about_product']} {row['category']}"
    for ch in sentence_splitting(txt):
        all_chunks.append(ch); metadata_chunks.append(row.to_dict())

## Step 4: IDs & Embeddings

In [ ]:
ids=[f"{m['product_id']}_chunk_{i}" for i,m in enumerate(metadata_chunks)]
model=SentenceTransformer('all-MiniLM-L6-v2')
embeddings=model.encode(all_chunks,batch_size=80,show_progress_bar=True)

## Step 5: Load FAISS & Batch Insert

In [ ]:
embedding_model=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store=FAISS.load_local('../vector_store/faiss_index',embedding_model,allow_dangerous_deserialization=True)
for i in range(0,len(all_chunks),80):
    vector_store.add_embeddings(list(zip(all_chunks[i:i+80],embeddings[i:i+80])),ids=ids[i:i+80],metadatas=metadata_chunks[i:i+80])

## Step 6: Advanced Search

In [ ]:
def advance_search(vector_store,model,query_text,category=None,min_price=None,max_price=None,rating=None,product_name=None,top_k=5):
    q=model.encode([query_text])[0]
    docs=vector_store.similarity_search_by_vector(q,k=top_k*5)
    out=[]
    for d in docs:
        m=d.metadata
        if category and category.lower() not in str(m.get('category','')).lower(): continue
        if min_price is not None and m.get('discounted_price',0)<min_price: continue
        if max_price is not None and m.get('discounted_price',1e9)>max_price: continue
        if rating is not None and m.get('rating',0)<rating: continue
        if product_name and product_name.lower() not in str(m.get('product_name','')).lower(): continue
        out.append(d)
        if len(out)>=top_k: break
    return out

## Step 7: Testing

In [ ]:
results=advance_search(vector_store,model,'best laptop for programming',top_k=3)
for r in results:
    print(r.metadata['product_name'],r.metadata['discounted_price'],r.metadata['rating'])

# Day 3 Complete
Advanced retrieval pipeline finished. Next: Connect an LLM to complete the RAG system.